In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score)
import joblib
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("🚀 COMPLETE MODEL TRAINING PIPELINE")
print("="*70)

# ============================================================================
# STEP 1: LOAD & EXPLORE DATA
# ============================================================================
print("\n" + "="*70)
print("📊 STEP 1: DATA LOADING")
print("="*70)

df = pd.read_csv('../data/user_nutritional_data.csv')

print(f"\n✅ Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"📋 Columns: {list(df.columns)}")
print(f"❓ Missing values: {df.isnull().sum().sum()}")
print(f"🔁 Duplicate rows: {df.duplicated().sum()}")

🚀 COMPLETE MODEL TRAINING PIPELINE

📊 STEP 1: DATA LOADING

✅ Dataset loaded: 2182 rows, 11 columns
📋 Columns: ['Gender', 'Age', 'Daily meals frequency', 'Physical exercise', 'Height', 'Weight', 'BMR', 'Carbs', 'Proteins', 'Fats', 'Calories']
❓ Missing values: 0
🔁 Duplicate rows: 84


In [2]:

# ============================================================================
# STEP 2: FEATURE ENGINEERING & TARGET CREATION
# ============================================================================
print("\n" + "="*70)
print("🔧 STEP 2: FEATURE ENGINEERING")
print("="*70)

# Remove duplicates
df_clean = df.drop_duplicates()
print(f"\n✅ Removed {len(df) - len(df_clean)} duplicates → {len(df_clean)} rows")

# Calculate BMI
df_clean['BMI'] = df_clean['Weight'] / ((df_clean['Height'] / 100) ** 2)
print(f"✅ BMI calculated (range: {df_clean['BMI'].min():.1f} - {df_clean['BMI'].max():.1f})")

# Calculate Calorie Balance
df_clean['Calorie_Balance'] = df_clean['Calories'] - df_clean['BMR']
print(f"✅ Calorie Balance calculated")

# Calculate Macro Ratios
df_clean['Protein_Ratio'] = (df_clean['Proteins'] * 4) / df_clean['Calories'] * 100
df_clean['Carbs_Ratio'] = (df_clean['Carbs'] * 4) / df_clean['Calories'] * 100
df_clean['Fats_Ratio'] = (df_clean['Fats'] * 9) / df_clean['Calories'] * 100
print(f"✅ Macro ratios calculated")

# ============================================================================
# CREATE HEALTH SCORE (0-100)
# ============================================================================
print("\n📌 Creating Health Score...")

health_score = np.full(len(df_clean), 100.0)

# BMI penalty (optimal: 18.5-24.9)
bmi_penalty = np.where(df_clean['BMI'] < 18.5, (18.5 - df_clean['BMI']) * 3, 0)
bmi_penalty += np.where(df_clean['BMI'] > 24.9, (df_clean['BMI'] - 24.9) * 2.5, 0)
health_score -= np.clip(bmi_penalty, 0, 30)

# Calorie balance penalty (±300 cal is healthy)
cal_penalty = np.abs(df_clean['Calorie_Balance']) / 50
health_score -= np.clip(cal_penalty, 0, 20)

# Protein ratio penalty (optimal: 15-30%)
protein_penalty = np.where(df_clean['Protein_Ratio'] < 15, 
                          (15 - df_clean['Protein_Ratio']) * 1.5, 0)
protein_penalty += np.where(df_clean['Protein_Ratio'] > 30, 
                           (df_clean['Protein_Ratio'] - 30) * 1, 0)
health_score -= np.clip(protein_penalty, 0, 15)

# Fat ratio penalty (optimal: 20-35%)
fat_penalty = np.where(df_clean['Fats_Ratio'] < 20, 
                      (20 - df_clean['Fats_Ratio']) * 1, 0)
fat_penalty += np.where(df_clean['Fats_Ratio'] > 35, 
                       (df_clean['Fats_Ratio'] - 35) * 1.5, 0)
health_score -= np.clip(fat_penalty, 0, 15)

# Exercise bonus (0-4 scale)
exercise_bonus = df_clean['Physical exercise'] * 2.5
health_score += exercise_bonus

# Meal frequency penalty (3 meals optimal)
meal_penalty = np.abs(df_clean['Daily meals frequency'] - 3) * 3
health_score -= meal_penalty

# Clip to 0-100
df_clean['Health_Score'] = np.clip(health_score, 0, 100)

print(f"✅ Health Score created")
print(f"   Range: {df_clean['Health_Score'].min():.2f} - {df_clean['Health_Score'].max():.2f}")
print(f"   Mean: {df_clean['Health_Score'].mean():.2f}")
print(f"   Median: {df_clean['Health_Score'].median():.2f}")

# ============================================================================
# CREATE HEALTH RISK (Binary: 0=Low, 1=High)
# ============================================================================
print("\n📌 Creating Health Risk Classification...")

# Use median as threshold for 50/50 split
threshold = df_clean['Health_Score'].median()
df_clean['Health_Risk'] = (df_clean['Health_Score'] < threshold).astype(int)

risk_counts = df_clean['Health_Risk'].value_counts().sort_index()
print(f"✅ Health Risk created (threshold: {threshold:.2f})")
print(f"   Low Risk (0): {risk_counts[0]} ({risk_counts[0]/len(df_clean)*100:.1f}%)")
print(f"   High Risk (1): {risk_counts[1]} ({risk_counts[1]/len(df_clean)*100:.1f}%)")

# Save processed data
df_clean.to_csv('../data/processed_nutritional_data.csv', index=False)
print(f"\n💾 Saved: ../data/processed_nutritional_data.csv")



🔧 STEP 2: FEATURE ENGINEERING

✅ Removed 84 duplicates → 2098 rows
✅ BMI calculated (range: 11.7 - 54.6)
✅ Calorie Balance calculated
✅ Macro ratios calculated

📌 Creating Health Score...
✅ Health Score created
   Range: 51.95 - 95.66
   Mean: 79.84
   Median: 84.50

📌 Creating Health Risk Classification...
✅ Health Risk created (threshold: 84.50)
   Low Risk (0): 1054 (50.2%)
   High Risk (1): 1044 (49.8%)

💾 Saved: ../data/processed_nutritional_data.csv


In [3]:
print("\n" + "="*70)
print("🎯 STEP 3: LINEAR REGRESSION TRAINING")
print("="*70)

# Features WITHOUT individual macros (to avoid multicollinearity)
feature_columns_linear = ['Gender', 'Age', 'Daily meals frequency', 'Physical exercise',
                         'Height', 'Weight', 'BMI', 'BMR', 'Calories']

X_linear = df_clean[feature_columns_linear]
y_linear = df_clean['Health_Score']

print(f"\n📊 Features: {len(feature_columns_linear)}")
print(f"   {feature_columns_linear}")

# Train-test split
X_train_lin, X_test_lin, y_train_lin, y_test_lin = train_test_split(
    X_linear, y_linear, test_size=0.2, random_state=42, 
    stratify=df_clean['Health_Risk']
)

print(f"\n📦 Split: Train={len(X_train_lin)}, Test={len(X_test_lin)}")

# Scale features
scaler_linear = StandardScaler()
X_train_lin_scaled = scaler_linear.fit_transform(X_train_lin)
X_test_lin_scaled = scaler_linear.transform(X_test_lin)

print(f"✅ Features scaled")

# Train model
linear_model = LinearRegression()
linear_model.fit(X_train_lin_scaled, y_train_lin)

# Evaluate
y_pred_lin = linear_model.predict(X_test_lin_scaled)
r2 = r2_score(y_test_lin, y_pred_lin)
mae = mean_absolute_error(y_test_lin, y_pred_lin)
rmse = np.sqrt(mean_squared_error(y_test_lin, y_pred_lin))

print(f"\n📈 Performance:")
print(f"   R² Score: {r2:.4f}")
print(f"   MAE: {mae:.4f}")
print(f"   RMSE: {rmse:.4f}")

# Test prediction
test_sample = pd.DataFrame({
    'Gender': [1], 'Age': [25], 'Daily meals frequency': [3],
    'Physical exercise': [3], 'Height': [175], 'Weight': [70],
    'BMI': [22.86], 'BMR': [1674], 'Calories': [2200]
})
test_scaled = scaler_linear.transform(test_sample)
test_pred = linear_model.predict(test_scaled)[0]
print(f"\n🧪 Test prediction (healthy male): {test_pred:.2f}")

# Save Linear Regression
joblib.dump(linear_model, '../models/linear_regression_model.pkl')
joblib.dump(scaler_linear, '../models/scaler.pkl')

feature_info = {
    'feature_columns': feature_columns_linear,
    'model_type': 'Linear Regression',
    'performance': {'r2_score': r2, 'mae': mae, 'rmse': rmse}
}
joblib.dump(feature_info, '../models/feature_info.pkl')

print(f"\n💾 Saved:")
print(f"   ✅ linear_regression_model.pkl")
print(f"   ✅ scaler.pkl")
print(f"   ✅ feature_info.pkl")



🎯 STEP 3: LINEAR REGRESSION TRAINING

📊 Features: 9
   ['Gender', 'Age', 'Daily meals frequency', 'Physical exercise', 'Height', 'Weight', 'BMI', 'BMR', 'Calories']

📦 Split: Train=1678, Test=420
✅ Features scaled

📈 Performance:
   R² Score: 0.8489
   MAE: 3.7900
   RMSE: 4.6423

🧪 Test prediction (healthy male): 87.94

💾 Saved:
   ✅ linear_regression_model.pkl
   ✅ scaler.pkl
   ✅ feature_info.pkl


In [4]:
print("\n" + "="*70)
print("🎯 STEP 4: LOGISTIC REGRESSION TRAINING")
print("="*70)

# Features WITH macros (for better classification)
feature_columns_logistic = ['Gender', 'Age', 'Daily meals frequency', 'Physical exercise',
                           'Height', 'Weight', 'BMI', 'BMR', 
                           'Carbs', 'Proteins', 'Fats', 'Calories']

X_logistic = df_clean[feature_columns_logistic]
y_logistic = df_clean['Health_Risk']

print(f"\n📊 Features: {len(feature_columns_logistic)}")
print(f"   {feature_columns_logistic}")

print(f"\n📦 Target distribution:")
print(f"   Low Risk (0): {(y_logistic==0).sum()}")
print(f"   High Risk (1): {(y_logistic==1).sum()}")

# Train-test split
X_train_log, X_test_log, y_train_log, y_test_log = train_test_split(
    X_logistic, y_logistic, test_size=0.2, random_state=42, stratify=y_logistic
)

print(f"\n📦 Split: Train={len(X_train_log)}, Test={len(X_test_log)}")

# Scale features
scaler_logistic = StandardScaler()
X_train_log_scaled = scaler_logistic.fit_transform(X_train_log)
X_test_log_scaled = scaler_logistic.transform(X_test_log)

print(f"✅ Features scaled")

# Train model
logistic_model = LogisticRegression(random_state=42, max_iter=1000)
logistic_model.fit(X_train_log_scaled, y_train_log)

# Evaluate
y_pred_log = logistic_model.predict(X_test_log_scaled)
y_proba_log = logistic_model.predict_proba(X_test_log_scaled)[:, 1]

accuracy = accuracy_score(y_test_log, y_pred_log)
precision = precision_score(y_test_log, y_pred_log)
recall = recall_score(y_test_log, y_pred_log)
f1 = f1_score(y_test_log, y_pred_log)
roc_auc = roc_auc_score(y_test_log, y_proba_log)

print(f"\n📈 Performance:")
print(f"   Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"   Precision: {precision:.4f}")
print(f"   Recall: {recall:.4f}")
print(f"   F1-Score: {f1:.4f}")
print(f"   ROC-AUC: {roc_auc:.4f}")

# Test predictions
print(f"\n🧪 Test Predictions:")

# Test 1: Healthy profile (should be LOW RISK)
test_low = pd.DataFrame({
    'Gender': [1], 'Age': [25], 'Daily meals frequency': [3],
    'Physical exercise': [3], 'Height': [175], 'Weight': [70],
    'BMI': [22.86], 'BMR': [1674], 'Carbs': [275], 'Proteins': [110],
    'Fats': [73], 'Calories': [2200]
})
test_low_scaled = scaler_logistic.transform(test_low)
pred_low = logistic_model.predict(test_low_scaled)[0]
proba_low = logistic_model.predict_proba(test_low_scaled)[0]
print(f"   Healthy profile: {pred_low} (0=Low, 1=High) - Proba: {proba_low[1]*100:.1f}%")

# Test 2: High risk profile (should be HIGH RISK)
test_high = pd.DataFrame({
    'Gender': [0], 'Age': [35], 'Daily meals frequency': [2],
    'Physical exercise': [0], 'Height': [160], 'Weight': [95],
    'BMI': [37.1], 'BMR': [1500], 'Carbs': [350], 'Proteins': [125],
    'Fats': [95], 'Calories': [2500]
})
test_high_scaled = scaler_logistic.transform(test_high)
pred_high = logistic_model.predict(test_high_scaled)[0]
proba_high = logistic_model.predict_proba(test_high_scaled)[0]
print(f"   High risk profile: {pred_high} (0=Low, 1=High) - Proba: {proba_high[1]*100:.1f}%")

if pred_low == 0 and pred_high == 1:
    print(f"\n   ✅ Both predictions CORRECT!")
else:
    print(f"\n   ⚠️ Check model - predictions may not be working correctly")

# Save Logistic Regression
joblib.dump(logistic_model, '../models/logistic_regression_model.pkl')
joblib.dump(scaler_logistic, '../models/scaler_logistic.pkl')

print(f"\n💾 Saved:")
print(f"   ✅ logistic_regression_model.pkl")
print(f"   ✅ scaler_logistic.pkl")


🎯 STEP 4: LOGISTIC REGRESSION TRAINING

📊 Features: 12
   ['Gender', 'Age', 'Daily meals frequency', 'Physical exercise', 'Height', 'Weight', 'BMI', 'BMR', 'Carbs', 'Proteins', 'Fats', 'Calories']

📦 Target distribution:
   Low Risk (0): 1054
   High Risk (1): 1044

📦 Split: Train=1678, Test=420
✅ Features scaled

📈 Performance:
   Accuracy: 0.9714 (97.14%)
   Precision: 0.9581
   Recall: 0.9856
   F1-Score: 0.9717
   ROC-AUC: 0.9906

🧪 Test Predictions:
   Healthy profile: 0 (0=Low, 1=High) - Proba: 6.2%
   High risk profile: 1 (0=Low, 1=High) - Proba: 99.8%

   ✅ Both predictions CORRECT!

💾 Saved:
   ✅ logistic_regression_model.pkl
   ✅ scaler_logistic.pkl


In [5]:
print("\n" + "="*70)
print("✅ TRAINING COMPLETE!")
print("="*70)

print(f"\n📊 Summary:")
print(f"   Dataset: {len(df_clean)} samples")
print(f"   Linear Regression R²: {r2:.4f}")
print(f"   Logistic Regression Accuracy: {accuracy:.4f}")
print(f"   Risk Split: {risk_counts[0]} Low / {risk_counts[1]} High")

print(f"\n📁 Files saved in ../models/:")
print(f"   ✅ linear_regression_model.pkl")
print(f"   ✅ logistic_regression_model.pkl")
print(f"   ✅ scaler.pkl")
print(f"   ✅ scaler_logistic.pkl")
print(f"   ✅ feature_info.pkl")

print(f"\n🚀 Ready to run: streamlit run app/app.py")
print("="*70)


✅ TRAINING COMPLETE!

📊 Summary:
   Dataset: 2098 samples
   Linear Regression R²: 0.8489
   Logistic Regression Accuracy: 0.9714
   Risk Split: 1054 Low / 1044 High

📁 Files saved in ../models/:
   ✅ linear_regression_model.pkl
   ✅ logistic_regression_model.pkl
   ✅ scaler.pkl
   ✅ scaler_logistic.pkl
   ✅ feature_info.pkl

🚀 Ready to run: streamlit run app/app.py
